In [0]:
from pyspark.sql.functions import col, count, when, mean

flights = spark.table("flights")

total_rows = flights.count()

missing_summary = flights.select(
    (
        100 * count(when(col("dep_delay").isNull(), 1)) / total_rows
    ).alias("pct_missing_dep_delay"),

    (
        100 * count(when(col("arr_delay").isNull(), 1)) / total_rows
    ).alias("pct_missing_arr_delay"),

    (
        100 * count(when(col("air_time").isNull(), 1)) / total_rows
    ).alias("pct_missing_air_time")
)

display(missing_summary)

In [0]:
# Mean of arr_delay
mean_arr_delay = flights.select(
    mean("arr_delay")
).collect()[0][0]

# Median of arr_delay
median_arr_delay = flights.approxQuantile(
    "arr_delay",
    [0.5],
    0.01
)[0]

print(f"Mean arr_delay: {mean_arr_delay}")
print(f"Median arr_delay: {median_arr_delay}")

In [0]:
# The median is a better fill value for arr_delay because flight delays contain
# extreme outliers. Very large delays can skew the mean upward, while the
# median is more resistant to unusually high or low values.

In [0]:
flights_dedup = flights.dropDuplicates([
    "year",
    "month",
    "day",
    "carrier",
    "flight"
])

print("Original rows:", flights.count())
print("Rows after removing duplicates:", flights_dedup.count())

In [0]:
# Calculate median dep_delay
median_dep_delay = flights_dedup.approxQuantile(
    "dep_delay",
    [0.5],
    0.01
)[0]

print("Median dep_delay:", median_dep_delay)

# Fill missing values
flights_filled = flights_dedup.fillna({
    "dep_delay": median_dep_delay
})

In [0]:
# Calculate Q1 and Q3
q1, q3 = flights_filled.approxQuantile(
    "arr_delay",
    [0.25, 0.75],
    0.01
)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [0]:
from pyspark.sql.functions import when

flights_cleaned = flights_filled.withColumn(
    "is_outlier",
    when(
        (col("arr_delay") < lower_bound) |
        (col("arr_delay") > upper_bound),
        True
    ).otherwise(False)
)

outlier_count = flights_cleaned.filter(
    col("is_outlier") == True
).count()

print("Number of outliers:", outlier_count)

In [0]:
# Validation 1: Row count
row_count = flights_cleaned.count()
status = "PASS" if row_count > 300000 else "FAIL"
print(f"[{status}] Row count > 300000: {row_count}")

# Validation 2: No NULL carriers
null_carriers = flights_cleaned.filter(
    col("carrier").isNull()
).count()

status = "PASS" if null_carriers == 0 else "FAIL"
print(f"[{status}] No NULL carriers: {null_carriers}")

# Validation 3: No NULL dep_delay
null_dep_delay = flights_cleaned.filter(
    col("dep_delay").isNull()
).count()

status = "PASS" if null_dep_delay == 0 else "FAIL"
print(f"[{status}] No NULL dep_delay: {null_dep_delay}")

# Validation 4: No negative air_time
negative_air_time = flights_cleaned.filter(
    col("air_time") < 0
).count()

status = "PASS" if negative_air_time == 0 else "FAIL"
print(f"[{status}] No negative air_time: {negative_air_time}")